In [117]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
import os
import pickle

In [118]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

In [119]:
def download_dataset(url, file):
    if not os.path.exists(file):
        parent_dir = os.path.dirname(file)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        response = requests.get(url)
        response.raise_for_status()
        with open(file, 'wb') as f:
            f.write(response.content)
    return file

In [120]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = pd.to_datetime(df.lpep_dropoff_datetime) - pd.to_datetime(df.lpep_pickup_datetime)
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    Categorical = ['PULocationID', 'DOLocationID']
    df[Categorical] = df[Categorical].astype(str)
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    return df

In [121]:
df_train = read_dataframe(download_dataset('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet','green_tripdata_2021-01.parquet' ))
df_val = read_dataframe(download_dataset('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet', 'green_tripdata_2021-02.parquet'))

In [122]:
Categorical =  ['PU_DO'] #['PULocationID', 'DOLocationID']
Numerical = ['trip_distance']

dv = DictVectorizer()
train_dicts = df_train[Categorical + Numerical].to_dict(orient = 'records')
x_train = dv.fit_transform(train_dicts)
val_dicts = df_val[Categorical + Numerical].to_dict(orient = 'records')
x_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [123]:
lr = LinearRegression().fit(x_train, y_train)
y_pred = lr.predict(x_val)
root_mean_squared_error(y_val, y_pred)

7.758715206809494

In [124]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv,lr), f_out)

FileNotFoundError: [Errno 2] No such file or directory: 'models/lin_reg.bin'